# Importing the important libraries


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
import numpy as np
from scipy.signal import detrend
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import re



# Mounting Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Colab Notebooks/F_Thesis'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/F_Thesis


# Configuration

In [3]:

OUTPUT_DIR = "energy_load_analysis_plots_final"
MIDDAY_HOURS = [12, 13, 14, 15, 16]
MIDNIGHT_HOURS = [1, 2, 3, 4, 5]
ORIGINAL_DATA = "CombinedEurope_clean_header.csv"
OPSD_DATA = "time_series_60min.csv"
WEATHER_DATA = "weather_data.csv"
OUTPUT_FILE = "Final_All_Countries_Dataset_Enhanced.csv"
ENERGY_PATH = "CombinedEurope_clean_header.csv"
OPSD_PATH = "time_series_60min.csv"
WEATHER_PATH = "weather_data.csv"
SAMPLE_MODE = False
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


# Data Preprocessing





1.   Data Loading


In [4]:
def validate_file_exists(file_path, file_type=""):
    if not os.path.exists(file_path):
        print(f" ERROR: {file_type} file not found: {file_path}")
        return False
    if os.path.getsize(file_path) == 0:
        print(f" ERROR: {file_type} file is empty: {file_path}")
        return False
    return True


def load_data_safely(file_path, sample_size=10000):
    print(f"  Loading: {os.path.basename(file_path)}")

    strategies = [
        lambda: pd.read_csv(file_path, low_memory=False, nrows=sample_size if sample_size else None),
        lambda: pd.read_csv(file_path, nrows=sample_size if sample_size else None),
        lambda: pd.read_csv(file_path, engine='python', nrows=sample_size if sample_size else None),
        lambda: pd.read_csv(file_path, encoding='latin1', nrows=sample_size if sample_size else None),
        lambda: pd.read_csv(file_path, encoding='utf-8', nrows=sample_size if sample_size else None),
    ]

    for i, strategy in enumerate(strategies, 1):
        try:
            df = strategy()
            print(f"   Success with strategy {i}")
            return df
        except Exception as e:
            continue

    raise ValueError(f"Could not load file {file_path}")


def get_country_codes(columns):
    countries = set()
    for col in columns:
        parts = col.split('_')
        if len(parts) >= 2 and len(parts[0]) == 2:
            if parts[0].isalpha() and parts[0].isupper():
                countries.add(parts[0])
    return sorted(list(countries))




2.   Column Mapping





In [5]:
def create_feature_patterns():
    return {
        'load': ['_load_actual_entsoe_transparency', '_load'],
        'solar': ['_solar_generation_actual', '_solar'],
        'wind_power': ['_wind_generation_actual', '_wind_onshore_generation_actual', '_wind'],
        'temperature': ['_temperature'],
        'wind_speed': ['_wind_speed', '_wind_velocity']
    }


def build_column_mapping(df_opsd, df_weather, all_countries):
    patterns = create_feature_patterns()
    mapping = {}
    country_features = {}

    for country in all_countries:
        for feature_type, feature_patterns in patterns.items():
            for pattern in feature_patterns:
                col_name = f"{country}{pattern}"
                if col_name in df_opsd.columns:
                    mapping[col_name] = f"{country}_{feature_type}"
                    country_features.setdefault(country, set()).add(feature_type)
                    break
                elif col_name in df_weather.columns:
                    mapping[col_name] = f"{country}_{feature_type}"
                    country_features.setdefault(country, set()).add(feature_type)
                    break

    return mapping, country_features


def identify_countries_with_features(country_features, all_countries):
    countries_with_wind = [c for c in all_countries if 'wind_power' in country_features.get(c, set())]
    countries_with_wind_speed = [c for c in all_countries if 'wind_speed' in country_features.get(c, set())]

    return countries_with_wind, countries_with_wind_speed




3. Data Extraction





In [6]:
def extract_feature_for_country(df_final, mapping, country, feature_type, df_opsd, df_weather):
    mapped_cols = [k for k, v in mapping.items() if v == f"{country}_{feature_type}"]
    if not mapped_cols:
        return df_final

    src_col = mapped_cols[0]
    if src_col in df_opsd.columns:
        df_final[f"{country}_{feature_type}"] = df_opsd[src_col]
    elif src_col in df_weather.columns:
        df_final[f"{country}_{feature_type}"] = df_weather[src_col]

    return df_final

def extract_all_features_for_country(df_final, mapping, country, df_opsd, df_weather):
    for feature in ['load', 'temperature', 'solar', 'wind_power']:
        df_final = extract_feature_for_country(df_final, mapping, country, feature, df_opsd, df_weather)

    mapped_cols = [k for k, v in mapping.items() if v == f"{country}_wind_speed"]
    if mapped_cols:
        src_col = mapped_cols[0]
        if src_col in df_weather.columns:
            df_final[f"{country}_wind_speed_m_s"] = df_weather[src_col]
            df_final[f"{country}_wind_speed_km_h"] = df_final[f"{country}_wind_speed_m_s"] * 3.6

    return df_final

4. Feature Engineering

In [7]:
def calculate_hdd_cdd(df_final, country):
    temp_col = f"{country}_temperature"
    if temp_col not in df_final.columns:
        return df_final

    temp_series = df_final[temp_col].fillna(method='ffill').fillna(method='bfill')
    df_final[f'{country}_hdd'] = np.maximum(0, 18 - temp_series)
    df_final[f'{country}_cdd'] = np.maximum(0, temp_series - 22)

    return df_final


def calculate_realistic_dew_point(df_final, country):
    temp_col = f"{country}_temperature"
    if temp_col not in df_final.columns:
        return df_final

    temp_series = df_final[temp_col].fillna(method='ffill').fillna(method='bfill')

    df_final['month'] = df_final['utc_timestamp'].dt.month

    seasonal_factor = np.where(
        df_final['month'].isin([6, 7, 8]), 0.85,
        np.where(df_final['month'].isin([12, 1, 2]), 0.70, 0.75)
    )

    base_difference = np.where(
        temp_series > 20, temp_series * 0.10,
        np.where(temp_series > 0, temp_series * 0.15, temp_series * 0.20)
    )

    df_final[f'{country}_dew_point'] = temp_series - (base_difference * seasonal_factor)
    df_final[f'{country}_dew_point'] = df_final[f'{country}_dew_point'].clip(
        lower=temp_series - 12,
        upper=temp_series - 1
    )

    df_final = df_final.drop(columns=['month'], errors='ignore')

    return df_final


def calculate_heating_intensity(df_final, country):
    if f'{country}_hdd' not in df_final.columns or f'{country}_temperature' not in df_final.columns:
        return df_final

    temp_series = df_final[f'{country}_temperature'].fillna(method='ffill').fillna(method='bfill')

    df_final['hour'] = df_final['utc_timestamp'].dt.hour

    heating_efficiency = np.where(
        temp_series < -10, 0.85,
        np.where(temp_series < 0, 0.92, 0.98)
    )

    time_factor = np.where(
        df_final['hour'].isin(list(range(0, 6)) + list(range(18, 24))), 1.25, 1.0
    )

    df_final[f'{country}_heating_intensity'] = (
        df_final[f'{country}_hdd'] * heating_efficiency * time_factor
    )

    df_final = df_final.drop(columns=['hour'], errors='ignore')

    return df_final


def calculate_cooling_intensity(df_final, country):
    if f'{country}_cdd' not in df_final.columns or f'{country}_temperature' not in df_final.columns:
        return df_final

    temp_series = df_final[f'{country}_temperature'].fillna(method='ffill').fillna(method='bfill')

    df_final['hour'] = df_final['utc_timestamp'].dt.hour
    df_final['weekday'] = df_final['utc_timestamp'].dt.weekday

    cooling_efficiency = np.where(
        temp_series > 35, 0.80,
        np.where(temp_series > 28, 0.90, 0.95)
    )

    business_hours = np.where(
        df_final['hour'].isin(range(9, 18)), 1.35,
        np.where(df_final['hour'].isin([7, 8, 18, 19]), 1.15, 0.85)
    )

    weekend_factor = np.where(df_final['weekday'] >= 5, 1.20, 1.0)

    df_final[f'{country}_cooling_intensity'] = (
        df_final[f'{country}_cdd'] * cooling_efficiency * business_hours * weekend_factor
    )

    df_final = df_final.drop(columns=['hour', 'weekday'], errors='ignore')

    return df_final


def add_wind_speed_features(df_final, country):
    wind_speed_km_h_col = f"{country}_wind_speed_km_h"

    if wind_speed_km_h_col in df_final.columns:
        print(f"     Wind speed for {country}: Column exists")
        return df_final

    alternative_patterns = [
        f"{country}_wind_speed",
        f"{country}_ws",
        f"{country}_windspeed",
        f"{country}_windspd",
    ]

    for pattern in alternative_patterns:
        if pattern in df_final.columns:
            df_final[wind_speed_km_h_col] = df_final[pattern]
            print(f"     Wind speed for {country}: Renamed {pattern} to standard format")
            return df_final

    wind_power_cols = [f"{country}_wind", f"{country}_wind_power", f"{country}_wind_generation"]

    for col in wind_power_cols:
        if col in df_final.columns:
            df_final[wind_speed_km_h_col] = 10 * np.cbrt(df_final[col].fillna(0) / 1000 + 1)
            print(f"     Wind speed for {country}: Synthetic from {col}")
            return df_final

    print(f"     Wind speed for {country}: No wind data available")
    df_final[wind_speed_km_h_col] = 0

    return df_final

5. Data Organization

In [8]:
def organize_columns_by_country(df, countries_with_wind):
    print("\n Organizing columns by country in specified order...")

    wind_cols = [col for col in df.columns if col.endswith('_wind') and not col.endswith('_wind_speed')]
    if wind_cols:
        rename_dict = {col: col.replace('_wind', '_wind_power_MW') for col in wind_cols}
        df = df.rename(columns=rename_dict)
        print(f"   Renamed {len(rename_dict)} wind columns: _wind → _wind_power_MW")

    wind_potential_cols = [col for col in df.columns if col.endswith('_wind_potential')]
    if wind_potential_cols:
        df = df.drop(columns=wind_potential_cols)
        print(f"   Dropped {len(wind_potential_cols)} wind_potential columns")

    ordered_cols = []

    for ts_col in ['utc_timestamp', 'cet_cest_timestamp']:
        if ts_col in df.columns:
            ordered_cols.append(ts_col)

    feature_order = [
        '_load', '_temperature', '_solar', '_dew_point',
        '_hdd', '_cdd', '_wind_power_MW', '_wind_speed_km_h',
        '_heating_intensity', '_cooling_intensity'
    ]

    for country in sorted(countries_with_wind):
        for feature in feature_order:
            col_name = f"{country}{feature}"
            if col_name in df.columns:
                ordered_cols.append(col_name)

    remaining_cols = [col for col in df.columns if col not in ordered_cols]
    if remaining_cols:
        ordered_cols.extend(remaining_cols)

    return df[ordered_cols]

6. Data Cleaning

In [9]:
def clean_dataset(df_final):
    if 'utc_timestamp' in df_final.columns and len(df_final) > 0:
        df_final = df_final[
            (df_final['utc_timestamp'].dt.year >= 2015) &
            (df_final['utc_timestamp'].dt.year <= 2020)
        ]
        print(f"  Filtered to {len(df_final):,} rows (2015-2020)")

    numeric_cols = df_final.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        df_final[numeric_cols] = df_final[numeric_cols].ffill().bfill()
        print(f"  Filled missing values in {len(numeric_cols)} numeric columns")

    return df_final


def optimize_dataset(df_final):
    initial_rows = len(df_final)
    df_final = df_final.drop_duplicates(subset=['utc_timestamp'] if 'utc_timestamp' in df_final.columns else None)
    removed = initial_rows - len(df_final)
    if removed > 0:
        print(f"  Removed {removed} duplicate rows")

    for col in df_final.columns:
        if df_final[col].dtype == 'float64':
            try:
                df_final[col] = df_final[col].astype(np.float32)
            except:
                pass

    return df_final

7. Summary

In [10]:
def print_dataset_summary(df_final, countries_with_wind):
    """Print comprehensive dataset summary"""
    print(f"\n DATASET SUMMARY:")
    print(f"  • Rows: {len(df_final):,}")
    print(f"  • Columns: {len(df_final.columns)}")
    print(f"  • Countries: {len(countries_with_wind)}")

    if 'utc_timestamp' in df_final.columns:
        print(f"  • Date range: {df_final['utc_timestamp'].min()} to {df_final['utc_timestamp'].max()}")

    print(f"\n COLUMN STRUCTURE (first 15 columns):")
    for i, col in enumerate(df_final.columns[:15], 1):
        print(f"  {i:2d}. {col}")

    if len(df_final.columns) > 15:
        print(f"  ... and {len(df_final.columns) - 15} more columns")

    if len(df_final) > 0 and countries_with_wind:
        first_country = countries_with_wind[0]
        print(f"\n SAMPLE DATA for {first_country} (first 2 rows):")

        sample_cols = ['utc_timestamp'] if 'utc_timestamp' in df_final.columns else []
        sample_cols.extend([
            f"{first_country}_load", f"{first_country}_temperature", f"{first_country}_solar",
            f"{first_country}_dew_point", f"{first_country}_hdd", f"{first_country}_cdd"
        ])

        sample_cols = [col for col in sample_cols if col in df_final.columns]
        if sample_cols:
            print(df_final[sample_cols].head(2).to_string(index=False))

8. Main Processing

In [11]:
def load_clean_and_enhance_dataset(energy_path, opsd_path, weather_path, sample_mode=False):

    print("\n STEP 1: Loading and validating files...")

    for path, name in zip([energy_path, opsd_path, weather_path], ["Energy", "OPSD", "Weather"]):
        if not validate_file_exists(path, name):
            return None

    sample_size = 10000 if sample_mode else None
    df_opsd = load_data_safely(opsd_path, sample_size)
    df_weather = load_data_safely(weather_path, sample_size)

    print("\n STEP 2: Parsing timestamps...")

    for df in [df_opsd, df_weather]:
        if 'utc_timestamp' in df.columns:
            df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'], utc=True, errors='coerce')

    print("\n STEP 3: Identifying countries and creating column mapping...")

    opsd_countries = get_country_codes(df_opsd.columns)
    weather_countries = get_country_codes(df_weather.columns)
    all_countries = sorted(set(opsd_countries + weather_countries))

    if not all_countries:
        print(" ERROR: No country codes found in data!")
        return None

    mapping, country_features = build_column_mapping(df_opsd, df_weather, all_countries)
    countries_with_wind, countries_with_wind_speed = identify_countries_with_features(country_features, all_countries)

    print(f"   Found {len(countries_with_wind)} countries with wind data")
    print(f"   Countries: {', '.join(countries_with_wind[:10])}")

    print("\n STEP 4: Creating integrated dataset...")

    df_final = pd.DataFrame()
    if 'utc_timestamp' in df_opsd.columns:
        df_final['utc_timestamp'] = df_opsd['utc_timestamp']
        if 'cet_cest_timestamp' in df_opsd.columns:
            df_final['cet_cest_timestamp'] = df_opsd['cet_cest_timestamp']

    for country in countries_with_wind:
        df_final = extract_all_features_for_country(df_final, mapping, country, df_opsd, df_weather)

    print(f"   Extracted features for {len(countries_with_wind)} countries")

    print("\n STEP 5: Cleaning base data...")
    df_final = clean_dataset(df_final)

    print("\n STEP 6: Calculating derived features...")
    print("  • HDD and CDD with realistic bases")
    print("  • Dew point with seasonal variation ")
    print("  • Heating intensity with efficiency factors ")
    print("  • Cooling intensity with business hours ")

    for country in countries_with_wind:
        df_final = calculate_hdd_cdd(df_final, country)
        df_final = calculate_realistic_dew_point(df_final, country)
        df_final = calculate_heating_intensity(df_final, country)
        df_final = calculate_cooling_intensity(df_final, country)
        df_final = add_wind_speed_features(df_final, country)

    print("   All derived features calculated with realistic physics")

    print("\n STEP 7: Organizing columns logically...")
    df_final = organize_columns_by_country(df_final, countries_with_wind)

    print("\n STEP 8: Final optimizations...")
    df_final = optimize_dataset(df_final)

    print("\n STEP 9: Saving enhanced dataset...")

    try:
        df_final.to_csv(OUTPUT_FILE, index=False)
        print(f" Saved to: {OUTPUT_FILE}")

        if os.path.exists(OUTPUT_FILE):
            file_size = os.path.getsize(OUTPUT_FILE) / (1024*1024)
            print(f"  File size: {file_size:.2f} MB")
    except Exception as e:
        print(f"  Error saving file: {e}")
        try:
            df_final.to_csv("fallback_output.csv", index=False)
            print("   Saved fallback file: fallback_output.csv")
        except:
            print("  Could not save any output file")

    print("\n" + "=" * 80)
    print("PROCESSING COMPLETE")
    print("=" * 80)

    print_dataset_summary(df_final, countries_with_wind)

    return df_final

# Execution

In [12]:
    df_final = load_clean_and_enhance_dataset(ORIGINAL_DATA, OPSD_DATA, WEATHER_DATA, sample_mode=False)
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"Final dataset saved to {OUTPUT_FILE}")


 STEP 1: Loading and validating files...
  Loading: time_series_60min.csv
   Success with strategy 1
  Loading: weather_data.csv
   Success with strategy 1

 STEP 2: Parsing timestamps...

 STEP 3: Identifying countries and creating column mapping...
   Found 28 countries with wind data
   Countries: AT, BE, BG, CH, CY, CZ, DE, DK, EE, ES

 STEP 4: Creating integrated dataset...
   Extracted features for 28 countries

 STEP 5: Cleaning base data...
  Filtered to 50,400 rows (2015-2020)
  Filled missing values in 103 numeric columns

 STEP 6: Calculating derived features...
  • HDD and CDD with realistic bases
  • Dew point with seasonal variation 
  • Heating intensity with efficiency factors 
  • Cooling intensity with business hours 
     Wind speed for AT: Synthetic from AT_wind_power
     Wind speed for BE: Synthetic from BE_wind_power
     Wind speed for BG: Synthetic from BG_wind_power
     Wind speed for CH: Synthetic from CH_wind_power
     Wind speed for CY: Synthetic from CY

# Hourly Data Creation

In [13]:
def create_hourly_load_data(df_final):
    """
    Extracts load-only data and ensures hourly frequency.
    """
    load_cols = [col for col in df_final.columns if col.endswith('_load')]
    if not load_cols:
        load_cols = [col for col in df_final.columns
                    if col not in ['utc_timestamp', 'cet_cest_timestamp']
                    and not col.endswith(('_solar', '_wind', '_temperature', '_dew_point', '_hdd', '_cdd',
     '_wind_power_MW', '_wind_speed_km_h' ))]

    df_hourly = df_final[['utc_timestamp'] + load_cols].copy()
    df_hourly.index = pd.to_datetime(df_hourly['utc_timestamp'])
    df_hourly = df_hourly.drop(columns=['utc_timestamp'])
    df_hourly = df_hourly.resample('h').mean()
    df_hourly = df_hourly.ffill().bfill()
    return df_hourly

# Segmenting The Load Means Time Interval

In [14]:
def calculate_segment_means(df_hourly, hours):
    print(f"Calculating segment means for hours: {hours}")

    df_filtered = df_hourly[df_hourly.index.hour.isin(hours)].copy()

    if len(df_filtered) == 0:
        print(f"  Warning: No data found for hours {hours}")
        return pd.DataFrame()

    df_filtered['DOY'] = df_filtered.index.dayofyear
    df_filtered['Segment'] = (df_filtered['DOY'] - 1) // 15 + 1
    df_filtered['Segment'] = df_filtered['Segment'].clip(upper=24)
    df_filtered['Year'] = df_filtered.index.year
    df_filtered['Segment_ID'] = df_filtered['Year'].astype(str) + '-' + df_filtered['Segment'].astype(str).str.zfill(2)

    numeric_cols = df_filtered.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col not in ['DOY', 'Segment', 'Year', 'Segment_ID']]

    df_means = df_filtered.groupby('Segment_ID')[numeric_cols + ['Segment', 'Year']].first()

    print(f"  Created {len(df_means)} segments")
    print(f"  Data shape: {df_means.shape}")

    return df_means


# Generating the Scater plots

In [15]:
def generate_plots(df_means, time_of_day):
    load_cols = [col for col in df_means.columns if col.endswith('_load')]

    if not load_cols:
        load_cols = [col for col in df_means.columns
                    if col not in ['Segment', 'Year', 'Segment_ID', 'DOY']
                    and not col.startswith('_')]

    print(f"Generating plots for {len(load_cols)} countries: {load_cols[:5]}...")

    for country_col in load_cols:
        plot_data = df_means.dropna(subset=[country_col]).copy()

        if plot_data.empty:
            print(f"  Skipping {country_col} - no valid data")
            continue

        plt.figure(figsize=(12, 6))

        years = sorted(plot_data['Year'].unique())

        colors = plt.cm.tab20(np.linspace(0, 1, len(years)))

        for year, color in zip(years, colors):
            year_data = plot_data[plot_data['Year'] == year]
            plt.scatter(year_data['Segment'].values,
                       year_data[country_col].values,
                       label=str(year),
                       alpha=0.7,
                       color=color,
                       s=50)

        country_name = country_col.replace('_load', '')

        plt.title(f'Mean Actual Load for {country_name} - {time_of_day} (15-Day Segments)',
                 fontsize=14, fontweight='bold')
        plt.xlabel('15-Day Segment of the Year (1 = Jan 1-15, ..., 24 = Dec 1-15)', fontsize=12)
        plt.ylabel('Mean Actual Load (MW)', fontsize=12)
        plt.xticks(np.arange(1, 25, 1))

        plt.grid(True, linestyle='--', alpha=0.6)

        if len(years) <= 10:
            plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
        else:
            plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left',
                      ncol=2, fontsize=8)

        plt.tight_layout()

        filename = f'{country_name.replace("/", "_")}_{time_of_day.lower().replace(" ", "_").replace(":", "")}_load_scatter.png'
        save_path = os.path.join(OUTPUT_DIR, filename)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

        plt.show()
        plt.close()

        print(f'  ✓ Saved plot for {country_name} to {save_path}')


In [16]:
df_final = load_clean_and_enhance_dataset(ORIGINAL_DATA, OPSD_DATA, WEATHER_DATA, sample_mode=False)
print("Starting data loading and cleaning...")
df_hourly = create_hourly_load_data(df_final)
print(f"Data loaded and cleaned. Total hourly records: {len(df_hourly)}")
print(f"Countries found: {list(df_hourly.columns)}")

# Midday Analysis
print("\nStarting Midday Analysis...")
df_midday_means = calculate_segment_means(df_hourly, MIDDAY_HOURS)
generate_plots(df_midday_means, 'Midday (12:00-16:00)')

# Midnight Analysis
print("\nStarting Midnight Analysis...")
df_midnight_means = calculate_segment_means(df_hourly, MIDNIGHT_HOURS)
generate_plots(df_midnight_means, 'Midnight (01:00-05:00)')

print("\nAnalysis complete. Plots saved to the 'energy_load_analysis_plots_final' directory.")


Output hidden; open in https://colab.research.google.com to view.

# Cross Correlation Matrix



*   **Preparing Data for Correlation**



In [17]:
def prepare_correlation_data(file_path):
    header_df = pd.read_csv(file_path, nrows=7, header=None)

    countries_row = header_df.iloc[0].fillna('')
    variables_row = header_df.iloc[1].fillna('')
    details_row = header_df.iloc[2].fillna('')
    orig_names_row = header_df.iloc[6].fillna('')

    new_cols = []
    for i in range(len(header_df.columns)):
        parts = []
        if countries_row.iloc[i]: parts.append(str(countries_row.iloc[i]))
        if variables_row.iloc[i]: parts.append(str(variables_row.iloc[i]))
        if details_row.iloc[i]: parts.append(str(details_row.iloc[i]))

        col_name = '_'.join(parts)
        if not col_name:
            col_name = str(orig_names_row.iloc[i])
        new_cols.append(col_name)

    df = pd.read_csv(file_path, skiprows=7, header=None, low_memory=False)
    df.columns = new_cols

    timestamp_col = None
    for col in ['cet_cest_timestamp', 'utc_timestamp']:
        if col in df.columns:
            timestamp_col = col
            break

    if timestamp_col is None:
        timestamp_col = df.columns[0]

    df['timestamp'] = pd.to_datetime(df[timestamp_col])
    df = df.set_index('timestamp')

    cols_to_drop = [c for c in ['utc_timestamp', 'cet_cest_timestamp'] if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.fillna(0)

    df['hour'] = df.index.hour
    df['segment_15d'] = (df.index.dayofyear - 1) // 15

    df_midday = df[(df['hour'] >= 12) & (df['hour'] < 16)].copy()
    df_midnight = df[(df['hour'] >= 1) & (df['hour'] < 5)].copy()

    return df_midday, df_midnight



*   **Compute_Cross_Correlation_Matrix**



In [18]:
def compute_cross_correlation_matrix(df_midday, df_midnight):
    load_cols = [col for col in df_midday.columns if '_load' in col]
    countries = sorted(list(set([col.split('_')[0] for col in load_cols])))

    print(f"Found {len(countries)} countries: {countries}")

    if not countries:
        print("ERROR: No countries detected. Check if your columns contain '_load'")
        return pd.DataFrame()

    results = []
    for country in countries:
        country_cols = [col for col in df_midday.columns if col.startswith(f"{country}_")]

        features = {}
        for col in country_cols:
            if '_load' in col:
                features['load'] = col
            elif '_solar' in col:
                features['solar'] = col
            elif '_wind_power' in col or '_wind_generation' in col:
                features['wind'] = col
            elif '_temperature' in col:
                features['temperature'] = col
            elif '_hdd' in col:
                features['hdd'] = col
            elif '_cdd' in col:
                features['cdd'] = col
            elif '_dew_point' in col or '_dew' in col:
                features['dew_point'] = col
            elif '_wind_speed_km_h' in col:
                features['wind_speed'] = col
            elif '_heating_intensity' in col or '_heating' in col:
                features['heating_intensity'] = col
            elif '_cooling_intensity' in col or '_cooling' in col:
                features['cooling_intensity'] = col

        midday_res = compute_row_metrics(df_midday, features)
        if midday_res:
            results.append({'Country': country, 'Time': 'Midday', **midday_res})

        midnight_res = compute_row_metrics(df_midnight, features)
        if midnight_res:
            results.append({'Country': country, 'Time': 'Midnight', **midnight_res})

    return pd.DataFrame(results)





*   **Computing Row metrics**




In [19]:
def compute_row_metrics(df, features):
    if 'load' not in features or features['load'] not in df.columns:
        return None

    df_clean = df.fillna(0)
    load_data = df_clean[features['load']].values
    if len(load_data) < 20:
        return None

    res = {}

    feature_mapping = {
        'solar': 'Solar',
        'wind': 'Wind',
        'temperature': 'Temperature',
        'hdd': 'HDD',
        'cdd': 'CDD',
        'dew_point': 'Dew_Point',
        'wind_speed': 'Wind_Speed',
        'heating_intensity': 'Heating_Intensity',
        'cooling_intensity': 'Cooling_Intensity'
    }

    def get_best_corr(feature_key):
        if feature_key in features and features[feature_key] in df_clean.columns:
            var_col = features[feature_key]
            var_data = df_clean[var_col].values[:len(load_data)]

            c1 = find_max_lag_correlation_scipy(load_data, var_data)
            c2 = find_max_lag_correlation_scipy(detrend(load_data), detrend(var_data))
            return c1 if abs(c1) > abs(c2) else c2
        return 0.0

    for feature_key, display_name in feature_mapping.items():
        res[display_name] = get_best_corr(feature_key)

    return res



*   **Finding_optimal_lag_correlations**




In [20]:
def find_max_lag_correlation_scipy(signal1, signal2, max_lag=48):
    if len(signal1) < 20 or len(signal2) < 20:
        return 0.0

    s1 = (signal1 - np.mean(signal1)) / (np.std(signal1) + 1e-8)
    s2 = (signal2 - np.mean(signal2)) / (np.std(signal2) + 1e-8)

    cross_corr = signal.correlate(s1, s2, mode='full', method='fft')
    lags = signal.correlation_lags(len(s1), len(s2), mode='full')

    lengths = np.minimum(np.arange(1, len(cross_corr) + 1),
                        np.arange(len(cross_corr), 0, -1))
    norm_corr = cross_corr / lengths

    mask = np.abs(lags) <= min(max_lag, len(s1) // 4)

    if any(mask):
        return norm_corr[mask][np.argmax(np.abs(norm_corr[mask]))]
    else:
        return 0.0

In [21]:
def generate_correlation_heatmaps(final_matrix):
    """
    Generates professional heatmaps from the correlation matrix.
    """
    for time_period in ['Midday', 'Midnight']:
        plot_df = final_matrix[final_matrix['Time'] == time_period].copy()
        if plot_df.empty:
            continue

        plot_df = plot_df.set_index('Country').drop(columns=['Time'])

        plt.figure(figsize=(14, 10))
        # Using 'RdBu_r' for signed values, with clear annotation and high contrast
        ax = sns.heatmap(plot_df, annot=True, cmap='RdBu_r', center=0, fmt=".2f",
                        linewidths=.5, vmin=-1, vmax=1,
                        annot_kws={"size": 10, "weight": "bold"}) # Bold text for clarity

        # Move x-axis labels to the top
        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top')

        plt.title(f'Load vs. Weather Features Correlation ({time_period})', fontsize=16, pad=40)
        plt.tight_layout()

        output_path = f"Correlation_Heatmap_{time_period}.png"
        plt.savefig(output_path, dpi=150)
        plt.close()
        print(f" Saved heatmap to: {output_path}")

In [25]:
# --- Execution Block ---
# Assuming df_midday and df_midnight are already prepared
df_midday, df_midnight = prepare_correlation_data(OUTPUT_FILE)

final_matrix = compute_cross_correlation_matrix(df_midday, df_midnight)

column_order = ['Country', 'Time', 'Solar', 'Wind', 'Temperature',
                'HDD', 'CDD', 'Dew_Point', 'Wind_Speed',
               'Heating_Intensity', 'Cooling_Intensity']

existing_cols = [col for col in column_order if col in final_matrix.columns]
final_matrix = final_matrix[existing_cols]

# # Generate Heatmaps
generate_correlation_heatmaps(final_matrix)

Found 28 countries: ['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
 Saved heatmap to: Correlation_Heatmap_Midday.png
 Saved heatmap to: Correlation_Heatmap_Midnight.png
